# Construct training and test datasets

In [8]:
! pip install polars pyfastatools --quiet

## Data sources
These data are not included in this repository due to their size. They can be obtained by following the links below.

### geNomad training dataset

Curated benchmarking databases used for geNomad by [Camargo et al., (2023) *Nat. Biotechnol.*](https://doi.org/10.1038/s41587-023-01953-y). Zenodo link: [https://zenodo.org/records/8049246](https://zenodo.org/records/8049246)

Contains:
* **`benchmark_data/train_test_sequences.fna`:** a dataset of artificially fragmented sequences built from curated bacterial, archaeal, eukaryotic, plasmid, and viral genomes
* **`benchmark_data/sequence_weights.tsv`:** ground-truth classifications of the `train_test_sequences.fna` dataset
* **`provirus_data/train_crf/mock_prophages.fna`:** a dataset of mock proviruses built from prokaryotic chromosome sequences and phage genomes
* **`provirus_data/train_crf/mock_prophages.tsv`:** descriptions of the structure of mock proviruses


### proGenomes/proMGE dataset
Bacterial and Archaeal genomes from [proGenomes v3](https://progenomes.embl.de/). This database contains "over 900,000 consistently annotated bacterial and archaeal genomes containing 4 billion genes from over 40,000 species. Strict quality controls are employed for the included genomes to enable accurate analyses."

For MGEs, the companion database to proGenomes, [proMGE](https://promge.embl.de/), will be used. "ProMGE is a resource for the retrieval of 6 Mobile Genetic Element (MGE) categories in 76K genomes (totalling to 2.4 million prokaryotic MGEs)." According to the [publication](https://doi.org/10.1093/nar/gkac163), MGEs were identified using a pangenomic approach from the genomes in proGenomes v3, and classified using marker gene profile HMMs.

Using these two databases, prokaryotic genomes will be obtained and MGE regions within them (including prophages) will be labeled.

## Dataset processing

In [1]:
import os
os.environ["POLARS_MAX_THREADS"] = str(50)
import polars as pl
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [2]:
from pathlib import Path
ROOT_DIR = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/training_data")
SOURCE_DIR = ROOT_DIR.joinpath("source_data")

### Organize and label the geNomad dataset

#### Combine the fragmented and mock provirus training datasets into one and label sequences as viral/nonviral

GeNomad excluded a small subset of sequences from their benchmarks becuase they failed their QC. Their IDs are in `genomad_supplementary_data/benchmark_data/excluded_from_benchmark.txt`, they will be excluded from the train/test datasets.

Also, to prevent data leakage that results from redundancy when splitting the dataset into test and train, fragments/contigs that were used to generate the mock provirus dataset will also be removed.

In [3]:
GENOMAD_SOURCE_DIR = SOURCE_DIR.joinpath("genomad_supplementary_data")
GENOMAD_EXCLUDE_PATH = GENOMAD_SOURCE_DIR.joinpath("benchmark_data/excluded_from_benchmark.txt")

In [4]:
! head {GENOMAD_EXCLUDE_PATH}

PQXD01000003.1|fragment_5
NZ_MDTU01000014.1|fragment_2
KX452698.1|fragment_2
KX452698.1|fragment_3
KX452698.1|fragment_4
KX452698.1|fragment_5
LN827560.1|fragment_3
LN827560.1|fragment_4
LN827560.1|fragment_5
LN827587.1|fragment_3


In [5]:
genomad_provirus_tsv = GENOMAD_SOURCE_DIR.joinpath("provirus_data/train_crf/mock_prophages.tsv")
genomad_provirus_df = pl.read_csv(genomad_provirus_tsv, separator="\t")
genomad_provirus_df

frag_id,host_id,phage_id,total_length,phage_coord
str,str,str,i64,str
"""fragment_1""","""JAADFU010000046.1""","""IMGVR_UViG_3300006880_000010|3300006880|Ga0075429_100000114""",5000,"""2500-2500"""
"""fragment_2""","""MGMF01000028.1""","""IMGVR_UViG_3300031965_000033|3300031965|Ga0326597_10000698""",5000,"""5000-5000"""
"""fragment_3""","""DIVV01000035.1""","""IMGVR_UViG_3300006916_000176|3300006916|Ga0070750_10000171""",5000,"""5000-5000"""
"""fragment_4""","""QKMY01000045.1_130007_268608""","""IMGVR_UViG_3300023180_000114|3300023180|Ga0255768_10000395""",5000,"""0-0"""
"""fragment_5""","""MIAA01000022.1""","""IMGVR_UViG_3300035670_000618|3300035670|Ga0310129_0001029""",5000,"""2500-2500"""
…,…,…,…,…
"""fragment_32996""","""DTQO01000121.1""","""IMGVR_UViG_3300005327_000002|3300005327|Ga0070658_10000094""",30000,"""0-30000"""
"""fragment_32997""","""JAAYUG010000049.1""","""IMGVR_UViG_3300006803_000004|3300006803|Ga0075467_10000259""",30000,"""0-30000"""
"""fragment_32998""","""JABMQR010000342.1""","""IMGVR_UViG_3300014913_000006|3300014913|Ga0164310_10000002""",30000,"""0-30000"""


In [6]:
GENOMAD_SEQ_PATH = GENOMAD_SOURCE_DIR.joinpath("benchmark_data/train_test_sequences.fna")

In [20]:
from pyfastatools import Parser

seqs_to_exclude = set()
host_parent_seqs = genomad_provirus_df.get_column("host_id").unique().to_list()
imgvr_seqs = genomad_provirus_df.get_column("phage_id").unique().to_list()

for record in Parser(GENOMAD_SEQ_PATH):
    parent_id = record.header.name.rpartition("|")[0]
    if parent_id in host_parent_seqs or imgvr_seqs:
        seqs_to_exclude.add(record.header.name)

seqs_to_exclude = list(seqs_to_exclude)

In [21]:
seqs_to_exclude[:5], seqs_to_exclude[-5:], len(seqs_to_exclude)

(['NZ_SZZM01000006.1|fragment_2',
  'SKXQ01000102.1|fragment_1',
  'BCSB01001895.1|fragment_1',
  'WHTO01000224.1|fragment_1',
  'SAO-all-SRF-0-8-5-00_k119_18130010|fragment_3'],
 ['IMGVR_UViG_3300017779_000007|3300017779|Ga0181395_1000014|fragment_1',
  'IMGVR_UViG_3300031787_000005|3300031787|Ga0315900_10000051|fragment_6',
  'CAITUV010000063.1|fragment_2',
  'MT889377.1|fragment_3',
  'circular_virus_516654|fragment_3'],
 1446182)

In [7]:
GENOMAD_PROVIRUS_PATH = GENOMAD_SOURCE_DIR.joinpath("provirus_data/train_crf/mock_prophages.fna")

In [8]:
FILTERED_GENOMAD_PATH = SOURCE_DIR.joinpath("genomad_dataset.fna")

In [ ]:
! cat {GENOMAD_SEQ_PATH} {GENOMAD_PROVIRUS_PATH} | seqkit grep -v -f {GENOMAD_EXCLUDE_PATH} -o {FILTERED_GENOMAD_PATH}

[INFO] 227 patterns loaded from file


In [ ]:
! grep -c ">" {GENOMAD_SEQ_PATH} {GENOMAD_PROVIRUS_PATH} {FILTERED_GENOMAD_PATH}

/storage2/scratch/kosmopoulos/projects/checkAMG/training_data/source_data/genomad_supplementary_data/benchmark_data/train_test_sequences.fna:1446182
/storage2/scratch/kosmopoulos/projects/checkAMG/training_data/source_data/genomad_supplementary_data/provirus_data/train_crf/mock_prophages.fna:33000
/storage2/scratch/kosmopoulos/projects/checkAMG/training_data/source_data/genomad_dataset.fna:1478955


In [ ]:
! grep ">" {FILTERED_GENOMAD_PATH} | head
! grep ">" {FILTERED_GENOMAD_PATH} | tail

>AE017199.1|fragment_1
>AE017199.1|fragment_2
>AE017199.1|fragment_3
>AE017199.1|fragment_4
>AE017199.1|fragment_5
>AE017199.1|fragment_6
>AE017199.1|fragment_7
>AE017199.1|fragment_8
>AE017199.1|fragment_9
>AE017199.1|fragment_10
grep: write error: Broken pipe
>fragment_32991
>fragment_32992
>fragment_32993
>fragment_32994
>fragment_32995
>fragment_32996
>fragment_32997
>fragment_32998
>fragment_32999
>fragment_33000


In [9]:
GENOMAD_SEQ_WEIGHTS = GENOMAD_SOURCE_DIR.joinpath("benchmark_data/sequence_weights.tsv")

In [10]:
genomad_train_df = pl.read_csv(
    GENOMAD_SEQ_WEIGHTS,
    separator="\t",
    has_header=False,
    new_columns=[
        "Contig", "reference_cluster", "classification",
        "weight_0", "weight_1", "weight_2", "weight_3", "weight_4", "weight_5"
        ]
    )

genomad_train_df = (
    genomad_train_df
    .rename({"classification": "Source"})
    .with_columns(
        pl.when(
            pl.col("Source") == "virus"
        )
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("True Positive"),
        pl.when(
            pl.col("Source") != "virus"
        )
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("True Negative")
    )
    .select(
        [
            "Contig",
            "Source",
            "True Positive",
            "True Negative"
        ]
    )
)

In [11]:
genomad_train_df

Contig,Source,True Positive,True Negative
str,str,bool,bool
"""AE017199.1|fragment_1""","""chromosome""",false,true
"""AE017199.1|fragment_2""","""chromosome""",false,true
"""AE017199.1|fragment_3""","""chromosome""",false,true
"""AE017199.1|fragment_4""","""chromosome""",false,true
"""AE017199.1|fragment_5""","""chromosome""",false,true
…,…,…,…
"""SPO-SPSG-MIX-0-8-5-00_k119_8164124|fragment_2""","""chromosome""",false,true
"""SPO-SPSG-MIX-0-8-5-00_k119_8164124|fragment_3""","""chromosome""",false,true
"""SPO-SPSG-MIX-0-8-5-00_k119_8186906|fragment_1""","""chromosome""",false,true


In [12]:
genomad_train_df.get_column("Source").value_counts()

Source,count
str,u64
"""virus""",603789
"""plasmid""",108857
"""chromosome""",733536


In [13]:
genomad_train_df.get_column("True Positive").value_counts()

True Positive,count
bool,u64
true,603789
false,842393


In [14]:
genomad_region_df = (
    genomad_provirus_df.with_columns(
        pl.col("phage_coord")
        .str.split_exact("-", 1)
        .alias("pc")
    )
    .with_columns(
        start_v=pl.col("pc").struct.field("field_0").cast(pl.Int64),
        end_v=pl.col("pc").struct.field("field_1").cast(pl.Int64),
    )
    .drop("pc")
)

no_viral = (
    genomad_region_df.filter(pl.col("start_v") == pl.col("end_v"))
    .select(
        contig=pl.col("frag_id"),
        start=pl.lit(0, dtype=pl.Int64),
        end=pl.col("total_length").cast(pl.Int64),
        contig_type=pl.lit("chromosome_only"),
        region_type=pl.lit("Host"),
        region_subtype=pl.lit("Host"),
    )
)

mixed = genomad_region_df.filter(pl.col("start_v") != pl.col("end_v"))

pre_host = (
    mixed.filter(pl.col("start_v") > 0)
    .select(
        contig=pl.col("frag_id"),
        start=pl.lit(0, dtype=pl.Int64),
        end=pl.col("start_v"),
        contig_type=pl.lit("chromosome_mixed"),
        region_type=pl.lit("Host"),
        region_subtype=pl.lit("Host"),
    )
)

viral = (
    mixed.select(
        contig=pl.col("frag_id"),
        start=pl.col("start_v"),
        end=pl.col("end_v"),
        contig_type=pl.lit("chromosome_mixed"),
        region_type=pl.lit("Viral"),
        region_subtype=pl.lit("Provirus"),
    )
)

post_host = (
    mixed.filter(pl.col("total_length") > pl.col("end_v"))
    .select(
        contig=pl.col("frag_id"),
        start=pl.col("end_v"),
        end=pl.col("total_length").cast(pl.Int64),
        contig_type=pl.lit("chromosome_mixed"),
        region_type=pl.lit("Host"),
        region_subtype=pl.lit("Host"),
    )
)

genomad_region_df = (
    pl.concat([no_viral, pre_host, viral, post_host], how="vertical_relaxed")
    .with_columns(
        pl.col("start").cast(pl.Int64),
        pl.col("end").cast(pl.Int64),
        pl.when(
            pl.col("region_type") == "Viral"
        )
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("True Positive"),
        pl.when(
            pl.col("region_type") != "Viral"
        )
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("True Negative")
    )
    .sort(["contig", "start", "end"])
)

In [15]:
genomad_region_df

contig,start,end,contig_type,region_type,region_subtype,True Positive,True Negative
str,i64,i64,str,str,str,bool,bool
"""fragment_1""",0,5000,"""chromosome_only""","""Host""","""Host""",false,true
"""fragment_10""",0,5000,"""chromosome_only""","""Host""","""Host""",false,true
"""fragment_100""",0,5000,"""chromosome_only""","""Host""","""Host""",false,true
"""fragment_1000""",0,900,"""chromosome_mixed""","""Viral""","""Provirus""",true,false
"""fragment_1000""",900,5000,"""chromosome_mixed""","""Host""","""Host""",false,true
…,…,…,…,…,…,…,…
"""fragment_9998""",0,10250,"""chromosome_mixed""","""Host""","""Host""",false,true
"""fragment_9998""",10250,12500,"""chromosome_mixed""","""Viral""","""Provirus""",true,false
"""fragment_9999""",0,5125,"""chromosome_mixed""","""Host""","""Host""",false,true


In [22]:
contig_to_len = {
    "contig": [],
    "length": []
}
for record in Parser(FILTERED_GENOMAD_PATH):
    contig_to_len["contig"].append(record.header.name)
    contig_to_len["length"].append(len(record.seq))

contig_to_len = pl.DataFrame(contig_to_len)

In [23]:
contig_to_len

contig,length
str,i64
"""AE017199.1|fragment_1""",5848
"""AE017199.1|fragment_2""",5326
"""AE017199.1|fragment_3""",11404
"""AE017199.1|fragment_4""",3983
"""AE017199.1|fragment_5""",27281
…,…
"""fragment_32996""",30000
"""fragment_32997""",30000
"""fragment_32998""",30000


In [24]:
genomad_merged_df = (
    genomad_train_df
    .with_columns(
        pl.when(
            pl.col("Source") == "virus"
        )
        .then(pl.lit("Viral"))
        .otherwise(
            pl.when(pl.col("Source") == "chromosome")
            .then(pl.lit("Host"))
            .otherwise(pl.lit("MGE"))
        )
        .alias("region_type")
    )
    .with_columns(
        pl.when(
            pl.col("Source") == "virus"
        )
        .then(pl.lit("Virus"))
        .otherwise(
            pl.when(pl.col("Source") == "chromosome")
            .then(pl.lit("Host"))
            .otherwise(pl.lit("Plasmid"))
        )
        .alias("region_subtype")
    )
    .with_columns(
        pl.when(
            pl.col("Source") == "virus"
        )
        .then(pl.lit("standalone_virus"))
        .otherwise(
            pl.when(pl.col("Source") == "chromosome")
            .then(pl.lit("chromosome_only"))
            .otherwise(pl.lit("standalone_mge"))
        )
        .alias("contig_type")
    )
    .with_columns(
        pl.lit(0, dtype=pl.Int64).alias("start"),
        pl.when(pl.col("region_type") == "Viral")
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("True Positive"),
        pl.when(pl.col("region_type") != "Viral")
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("True Negative")
    )
    .join(
        contig_to_len,
        left_on="Contig",
        right_on="contig",
        how="left"
    )
    .rename({"length": "end", "Contig": "contig"})
    .select(
        [
            "contig", "start", "end", "contig_type", "region_type", "region_subtype", "True Positive", "True Negative"
        ]
    )
)

In [25]:
genomad_merged_df = pl.concat(
    [genomad_merged_df, genomad_region_df],
    how="vertical"
)

In [26]:
genomad_merged_df

contig,start,end,contig_type,region_type,region_subtype,True Positive,True Negative
str,i64,i64,str,str,str,bool,bool
"""AE017199.1|fragment_1""",0,5848,"""chromosome_only""","""Host""","""Host""",false,true
"""AE017199.1|fragment_2""",0,5326,"""chromosome_only""","""Host""","""Host""",false,true
"""AE017199.1|fragment_3""",0,11404,"""chromosome_only""","""Host""","""Host""",false,true
"""AE017199.1|fragment_4""",0,3983,"""chromosome_only""","""Host""","""Host""",false,true
"""AE017199.1|fragment_5""",0,27281,"""chromosome_only""","""Host""","""Host""",false,true
…,…,…,…,…,…,…,…
"""fragment_9998""",0,10250,"""chromosome_mixed""","""Host""","""Host""",false,true
"""fragment_9998""",10250,12500,"""chromosome_mixed""","""Viral""","""Provirus""",true,false
"""fragment_9999""",0,5125,"""chromosome_mixed""","""Host""","""Host""",false,true


In [27]:
GENOMAD_REGIONS_OUTPUT = SOURCE_DIR.joinpath("genomad_dataset_regions.tsv")

In [27]:
genomad_merged_df.write_csv(
    GENOMAD_REGIONS_OUTPUT,
    separator="\t"
)

### Organize and label the ProGenomes dataset

#### Load MGE information
Obtained from the proMGE [MGE profiles table](https://promge.embl.de/download.cgi).

**NOTE**: From the ProMGE website:
>MGE boundaries represent upper limits of genomic regions that harbour one or more MGEs of same or different types.

This means some regions labeled "Phage" for example, may still contain non-phage MGEs or host sequence. To avoid including these mixed regions and labelling them all as "Phage", **CheckAMG's ambiguous viral region calculations will be used to mark additional regions as nonviral**, below. So the region labels produced by the `gen_progenomes_data.py` script that is being used below should not be considered final.

In [28]:
PROGENOMES_SORUCE_DIR = Path("/storage2/scratch/kosmopoulos/databases/progenomes_promge")
PROMGE_TBL = PROGENOMES_SORUCE_DIR.joinpath("mges.txt")

In [29]:
pro_mge = pl.read_csv(PROMGE_TBL, separator="\t")
pro_mge = pro_mge.with_columns(
    pl.col("mge_genome_position").str.split_exact(":", 1).struct.field("field_0").alias("contig"),
    pl.col("mge_genome_position").str.split_exact(":", 1).struct.field("field_1").alias("position")
)
pro_mge = pro_mge.with_columns(
    pl.col("position").str.split_exact("-", 1).struct.field("field_0").cast(pl.Int64).alias("start"),
    pl.col("position").str.split_exact("-", 1).struct.field("field_1").cast(pl.Int64).alias("end")
).drop("mge_genome_position", "position")
start_cols = ["contig", "start", "end"]
pro_mge = pro_mge.select(start_cols + [col for col in pro_mge.columns if col not in start_cols])

In [30]:
pro_mge

contig,start,end,mge_length,number_of_proteins,phage_genes,conjugation_genes,recombinase,genome,mge_category,count,specI,kingdom,phylum,class,order,family,genus,species
str,i64,i64,i64,i64,i64,i64,str,str,str,i64,str,str,str,str,str,str,str,str
"""562.SAMN04376786.LVNY01000045""",952,4426,3474,6,0,0,"""rve""","""562.SAMN04376786""","""IS_Tn""",1,"""specI_v3_Cluster95""","""Bacteria""","""Proteobacteria""","""Gammaproteobacteria""","""Enterobacterales""","""Enterobacteriaceae""","""Escherichia""","""Escherichia coli"""
"""1870951.SAMN04252929.LPPF01000080""",108,84399,84291,79,0,0,"""ser_ce,Phage_integrase,Xer""","""1870951.SAMN04252929""","""IS_Tn""",1,"""specI_v3_Cluster77""","""Bacteria""","""Proteobacteria""","""Gammaproteobacteria""","""Enterobacterales""","""Enterobacteriaceae""","""Enterobacter""","""Enterobacter roggenkampii"""
"""1313.SAMEA1032643.CNPJ02000077""",205410,234256,28846,21,0,0,"""DDE_3,DDE_Tnp_IS66""","""1313.SAMEA1032643""","""IS_Tn""",2,"""specI_v3_Cluster282""","""Bacteria""","""Firmicutes""","""Bacilli""","""Lactobacillales""","""Streptococcaceae""","""Streptococcus""","""Streptococcus pneumoniae"""
"""1313.SAMEA1032643.CNPJ02000077""",179268,184242,4974,3,0,0,"""huh_y1""","""1313.SAMEA1032643""","""IS_Tn""",1,"""specI_v3_Cluster282""","""Bacteria""","""Firmicutes""","""Bacilli""","""Lactobacillales""","""Streptococcaceae""","""Streptococcus""","""Streptococcus pneumoniae"""
"""1313.SAMEA1032643.CNPJ02000077""",155048,175733,20685,25,0,0,"""Int_Tn916,Phage_integrase,Transposase_mut""","""1313.SAMEA1032643""","""IS_Tn""",1,"""specI_v3_Cluster282""","""Bacteria""","""Firmicutes""","""Bacilli""","""Lactobacillales""","""Streptococcaceae""","""Streptococcus""","""Streptococcus pneumoniae"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""1117645.SAMN04254539.CP016370""",1319289,1320179,890,1,0,0,"""Xer""","""1117645.SAMN04254539""","""Cellular""",1,"""specI_v3_Cluster2037""","""Bacteria""","""Bacteroidetes""","""Flavobacteriia""","""Flavobacteriales""","""Flavobacteriaceae""","""Elizabethkingia""","""Elizabethkingia anophelis"""
"""630.SAMEA980221.CQBI01000053""",98,22707,22609,20,0,0,"""Xer""","""630.SAMEA980221""","""Cellular""",1,"""specI_v3_Cluster1093""","""Bacteria""","""Proteobacteria""","""Gammaproteobacteria""","""Enterobacterales""","""Yersiniaceae""","""Yersinia""","""Yersinia enterocolitica"""
"""135461.SAMN04419588.LRFK01000012""",738616,741508,2892,3,0,0,"""Xer""","""135461.SAMN04419588""","""Cellular""",1,"""specI_v3_Cluster278""","""Bacteria""","""Firmicutes""","""Bacilli""","""Bacillales""","""Bacillaceae""","""Bacillus""","""Bacillus subtilis"""


### Construct and label MGEs/non-MGEs from ProMGE & ProGenomes3 databases
Makes 'bins' of sequences using MGEs (transposons and ICEs only) and prophages as marked in the ProMGE database table (min length 1000 bp):

1. **'Standalone MGE'**: Full sequence is an MGE, no host chromosome or phage sequences
2. **'Standalone Virus'**: Full sequence is a prophage, no host chromosome or MGE sequences
3. **'Chromosome Only'**: Full sequence is a host chromosome, made by removing MGEs and prophages regions from contigs
4. **'Mixed Chromosome'**: Contig containing a mix of host chromosome, MGE, and/or prophage regions, sometimes all three

The script `gen_progenomes_data.py` located in `accessory_scripts` will be used for this.

In [31]:
SCRIPTS_DIR = Path("./accessory_scripts")

In [32]:
PROGENOMES_FASTA = PROGENOMES_SORUCE_DIR.joinpath("progenomes3.contigs.representatives.fasta")
PROGENOMES_HEADERS = PROGENOMES_SORUCE_DIR.joinpath("progenomes3.contigs.representatives.headers.txt")
MIN_SEQ_LEN = 1000
PROGENOMES_OUT_DIR = SOURCE_DIR.joinpath("progenomes_mge")
os.makedirs(PROGENOMES_OUT_DIR, exist_ok=True)

In [ ]:
! python3 \
    {SCRIPTS_DIR.joinpath("gen_progenomes_data.py")} \
    --mge_table {PROMGE_TBL} \
    --fasta {PROGENOMES_FASTA} \
    --headers {PROGENOMES_HEADERS} \
    --min_len {MIN_SEQ_LEN} \
    --outdir {PROGENOMES_OUT_DIR} \
    --threads 10

2026-07-06 14:45:31 INFO:__main__: Reading MGE table: /storage2/scratch/kosmopoulos/databases/progenomes_promge/mges.txt
2026-07-06 14:45:32 INFO:__main__: Initial MGE table: 2,096,350 rows
2026-07-06 14:45:32 INFO:__main__: Filtering MGEs by minimum length: 1,000 bp
2026-07-06 14:45:32 INFO:__main__: After length filter: 1,875,293 rows
2026-07-06 14:45:32 INFO:__main__: Filtering by headers from: /storage2/scratch/kosmopoulos/databases/progenomes_promge/progenomes3.contigs.representatives.headers.txt
2026-07-06 14:45:36 INFO:__main__: 4,305,248 headers loaded.
2026-07-06 14:45:50 INFO:__main__: After header filter: 60,499 rows
2026-07-06 14:45:50 INFO:__main__: After MGE type filter: 43,655 rows
2026-07-06 14:45:50 INFO:__main__: Strictly partitioning MGEs into two non-overlapping bins of contigs...
2026-07-06 14:45:50 INFO:__main__: Standalone MGE bin contigs: 9,915, chrom+MGE bin contigs: 9,915
2026-07-06 14:45:50 INFO:__main__: Reading 19,830 required contigs from FASTA...
2026-07-

In [33]:
! wc -l {PROGENOMES_OUT_DIR.joinpath("chromosome_only_regions.tsv")}
! wc -l {PROGENOMES_OUT_DIR.joinpath("chromosome_mixed_regions.tsv")}
! wc -l {PROGENOMES_OUT_DIR.joinpath("standalone_mge_regions.tsv")}
! wc -l {PROGENOMES_OUT_DIR.joinpath("standalone_virus_regions.tsv")}

12074 /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/source_data/progenomes_mge/chromosome_only_regions.tsv
61729 /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/source_data/progenomes_mge/chromosome_mixed_regions.tsv
8109 /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/source_data/progenomes_mge/standalone_mge_regions.tsv
863 /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/source_data/progenomes_mge/standalone_virus_regions.tsv


In [34]:
! grep -c ">" {PROGENOMES_OUT_DIR.joinpath("chromosome_only.fasta")}
! grep -c ">" {PROGENOMES_OUT_DIR.joinpath("chromosome_mixed.fasta")}
! grep -c ">" {PROGENOMES_OUT_DIR.joinpath("standalone_mge.fasta")}
! grep -c ">" {PROGENOMES_OUT_DIR.joinpath("standalone_virus.fasta")}

4754
9915
7856
862


In [33]:
PROGENOMES_PROCESSED_FASTA = SOURCE_DIR.joinpath("progenomes_dataset.fna")

In [ ]:
! cat \
    {PROGENOMES_OUT_DIR.joinpath("chromosome_only.fasta")} \
    {PROGENOMES_OUT_DIR.joinpath("chromosome_mixed.fasta")} \
    {PROGENOMES_OUT_DIR.joinpath("standalone_mge.fasta")} \
    {PROGENOMES_OUT_DIR.joinpath("standalone_virus.fasta")} \
    > {PROGENOMES_PROCESSED_FASTA}

In [38]:
! grep -c ">" {PROGENOMES_PROCESSED_FASTA}
! grep ">" {PROGENOMES_PROCESSED_FASTA} | head
! grep ">" {PROGENOMES_PROCESSED_FASTA} | tail

23387
>1311774.SAMN01828154.KB849796~1-409563~chromosome_only
>1349766.SAMD00046732.BCTG01000028~1-61065~chromosome_only
>1266908.SAMN02441754.AQPB01000057~1-112054~chromosome_only
>1302690.SAMN05444008.FQUO01000034~1-111~chromosome_only
>1288484.SAMN02470573.APCS01000112~1-5873~chromosome_only
>1232458.SAMD00008692.BAIL02000071~1-70466~chromosome_only
>1218093.SAMD00045732.BCPU01000043~1-20331~chromosome_only
>1329909.SAMN02471719.ATHO01000164~1-1~chromosome_only
>1261070.SAMN02436829.KE346614~1-218~chromosome_only
>1354027.SAMN02255022.AUOZ01000044~1-298~chromosome_only
grep: write error: Broken pipe
>1348626.SAMD00046991.BCVE01000022~35506-62197~standalone_virus
>1122922.SAMN02441525.AUFO01000001~92006-679312~standalone_virus
>1028490.SAMEA4028762.FLYG01000003~637465-679655~standalone_virus
>1028490.SAMEA4028762.FLYG01000003~41124-55058~standalone_virus
>1028490.SAMEA4028762.FLYG01000003~1402113-1469121~standalone_virus
>1028490.SAMEA4028762.FLYG01000003~2436018-2488736~standalone_v

In [34]:
region_paths = [
    PROGENOMES_OUT_DIR.joinpath("chromosome_only_regions.tsv"),
    PROGENOMES_OUT_DIR.joinpath("chromosome_mixed_regions.tsv"),
    PROGENOMES_OUT_DIR.joinpath("standalone_mge_regions.tsv"),
    PROGENOMES_OUT_DIR.joinpath("standalone_virus_regions.tsv")
]
region_dfs = {}

for region_path in region_paths:
    name = os.path.basename(region_path).replace(".tsv", "").replace("_regions", "")
    region_df = pl.read_csv(region_path, separator="\t")
    region_df = region_df.with_columns(
        pl.col("contig").str.split_exact("~", 1).struct.field("field_0").alias("orig_id")
    )
    region_dfs[name] = region_df

progenomes_region_df = pl.concat(region_dfs.values(), how="vertical")
del region_dfs

progenomes_region_df = progenomes_region_df.join(
    pro_mge.select(["kingdom", "phylum", "class", "order", "family", "genus", "contig"]).rename({"contig": "orig_id"}),
    on="orig_id",
    how="left"
).rename({"kingdom": "domain"}).unique()

progenomes_region_df = progenomes_region_df.with_columns(
    pl.when(pl.col("contig_type") == "standalone_virus")
    .then(pl.lit("standalone_virus"))
    .otherwise(pl.col("contig_type"))
    .alias("contig_type"),
    pl.when(pl.col("region_type") == "Viral")
    .then(pl.lit(True))
    .otherwise(pl.lit(False))
    .alias("True Positive"),
    pl.when(pl.col("region_type") != "Viral")
    .then(pl.lit(True))
    .otherwise(pl.lit(False))
    .alias("True Negative"),
)

progenomes_region_df = (
    progenomes_region_df
    .unique()
    .sort(["contig_type", "region_type", "region_subtype", "contig", "start"])
)

In [35]:
progenomes_region_df

contig,start,end,contig_type,region_type,region_subtype,orig_id,domain,phylum,class,order,family,genus,True Positive,True Negative
str,i64,i64,str,str,str,str,str,str,str,str,str,str,bool,bool
"""1423786.SAMN02369507.AZFZ01000109~1-4322~chromosome_mixed""",0,1453,"""chromosome_mixed""","""Host""","""Host""","""1423786.SAMN02369507.AZFZ01000109""","""Bacteria""","""Firmicutes""","""Bacilli""","""Lactobacillales""","""Lactobacillaceae""","""Lactobacillus""",false,true
"""1423786.SAMN02369507.AZFZ01000109~1-4322~chromosome_mixed""",4169,4322,"""chromosome_mixed""","""Host""","""Host""","""1423786.SAMN02369507.AZFZ01000109""","""Bacteria""","""Firmicutes""","""Bacilli""","""Lactobacillales""","""Lactobacillaceae""","""Lactobacillus""",false,true
"""1423786.SAMN02369507.AZFZ01000115~1-3475~chromosome_mixed""",0,396,"""chromosome_mixed""","""Host""","""Host""","""1423786.SAMN02369507.AZFZ01000115""","""Bacteria""","""Firmicutes""","""Bacilli""","""Lactobacillales""","""Lactobacillaceae""","""Lactobacillus""",false,true
"""1423786.SAMN02369507.AZFZ01000115~1-3475~chromosome_mixed""",2551,3475,"""chromosome_mixed""","""Host""","""Host""","""1423786.SAMN02369507.AZFZ01000115""","""Bacteria""","""Firmicutes""","""Bacilli""","""Lactobacillales""","""Lactobacillaceae""","""Lactobacillus""",false,true
"""1423786.SAMN02369507.AZFZ01000133~1-2580~chromosome_mixed""",0,140,"""chromosome_mixed""","""Host""","""Host""","""1423786.SAMN02369507.AZFZ01000133""","""Bacteria""","""Firmicutes""","""Bacilli""","""Lactobacillales""","""Lactobacillaceae""","""Lactobacillus""",false,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""1423772.SAMN02369407.AYYN01000140~178938-247278~standalone_virus""",0,68340,"""standalone_virus""","""Viral""","""Provirus""","""1423772.SAMN02369407.AYYN01000140""","""Bacteria""","""Firmicutes""","""Bacilli""","""Lactobacillales""","""Lactobacillaceae""","""Lactobacillus""",true,false
"""1423775.SAMN02369463.AZDZ01000022~27730-319079~standalone_virus""",0,291349,"""standalone_virus""","""Viral""","""Provirus""","""1423775.SAMN02369463.AZDZ01000022""","""Bacteria""","""Firmicutes""","""Bacilli""","""Lactobacillales""","""Lactobacillaceae""","""Lactobacillus""",true,false
"""1423779.SAMN02369408.AZGE01000005~93-45265~standalone_virus""",0,45172,"""standalone_virus""","""Viral""","""Provirus""","""1423779.SAMN02369408.AZGE01000005""","""Bacteria""","""Firmicutes""","""Bacilli""","""Lactobacillales""","""Lactobacillaceae""","""Lactobacillus""",true,false


In [36]:
PROGENOMES_REGIONS_OUTPUT = SOURCE_DIR.joinpath("progenomes_dataset_regions.tsv")

In [39]:
progenomes_region_df.write_csv(
    PROGENOMES_REGIONS_OUTPUT,
    separator="\t"
)

## Run CheckAMG annotate v1.1 on geNomad and proGenomes datasets to get features for model training

* Using CheckAMG database `CheckAMG_annotate_db_v1.1_20260316`
* Will remove filtered sequences later
* Using `--min-len 1` and `--min-orf 1` to allow smaller standalone MGE sequences

### Run CheckAMG annotate on the geNomad dataset

    nohup checkamg annotate \
        --db-dir ./CheckAMG_annotate_db_v1.1_20260316 \
        --output /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/checkamg_annotate_outputs_1.1/checkamg_annotate_genomad_dataset \
        --input-contigs /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/source_data/genomad_dataset.fna \
        --min-len 1 \
        --min-orf 1 \
        --cov-fraction 0.4 \
        --bitscore 60 \
        --window-size 5000 \
        --threads 100 \
        --mem 1200 \
        --save-to-parquet \
        --keep-full-hmm-results \
        > ./logs/checkamg_annotate/CheckAMG_annotate_genomad_dataset.log &

### Run CheckAMG annotate on the proGenomes dataset

    nohup checkamg annotate \
        --db-dir ./CheckAMG_annotate_db_v1.1_20260316 \
        --output /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/checkamg_annotate_outputs_1.1/checkamg_annotate_progenomes_dataset \
        --input-contigs /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/source_data/progenomes_dataset.fna \
        --min-len 1 \
        --min-orf 1 \
        --cov-fraction 0.4 \
        --bitscore 60 \
        --window-size 5000 \
        --threads 100 \
        --mem 1200 \
        --save-to-parquet \
        --keep-full-hmm-results \
        > ./logs/checkamg_annotate/CheckAMG_annotate_progenomes_dataset.log &

### Load and format the CheckAMG results

In [37]:
CHECKAMG_OUTPUT_DIR = ROOT_DIR.joinpath("processing/checkamg_annotate_outputs_1.1")

In [38]:
import gc

def load_data(final_file, context_file, regions_tsv, modify_labels=False):
    gc.collect()

    # 1) Lazy‐load your protein hits + context
    hits_lazy = (
        pl.scan_parquet(final_file).unique()
          .join(
              pl.scan_parquet(context_file).drop("contig"),
              left_on="Protein", right_on="protein", how="left"
          )
          .join(
              pl.scan_parquet(context_file).unique()
                .rename({"contig": "Contig"})
                .select("Contig")
                .unique(),
              on="Contig", how="inner"
          )
    )
    
    # 2) Lazy‐prepare region table
    regions = pl.scan_csv(regions_tsv, separator="\t").unique()
    regions_lazy = (
        regions.lazy()
        .filter(pl.col("region_type").is_in(["Viral","Host","MGE"]))
        .rename({
            "contig": "Contig",
            "start": "region_start",
            "end": "region_end"
        })
        .select(["Contig","region_start","region_end","region_type","contig_type"])
        .with_columns(
            pl.when(pl.col("region_type") == "Viral").then(pl.lit("Virus"))
            .otherwise(pl.col("region_type")).alias("region_type")
        )
    )
    
    # 3) Interval‐join & pick longest overlap per Protein
    region_hits_lazy = (
        hits_lazy
          .join(regions_lazy, on="Contig", how="inner")
          .filter(
              (pl.col("contig_pos_end") >= pl.col("region_start")) &
              (pl.col("contig_pos_start") <= pl.col("region_end"))
          )
          .with_columns(
              (pl.col("region_end") - pl.col("region_start")).alias("region_length")
          )
          .sort("region_length", descending=True)
          .group_by("Protein")
          .agg([
              pl.first("region_type").alias("Source"),
              pl.first("contig_type").alias("region_contig_type")
          ])
    )

    # 4) Inner‐join back & compute TP/TN/region_contig_type
    final_lazy = (
        hits_lazy
        .join(region_hits_lazy, on="Protein", how="inner")
        .with_columns([
            (pl.col("Source") == "Virus").alias("True Positive"),
            (~(pl.col("Source") == "Virus")).alias("True Negative")
        ])
        .with_columns(
            (pl.col("contig_pos_start") - pl.col("contig_pos_end")).abs().alias("length"),
            pl.when(pl.col("Source") == "Virus").then(pl.lit("Viral"))
            .otherwise(pl.lit("Nonviral")).alias("region_label"),
        )
    ).select(
        [
            # identifiers
            'Protein',
            'Contig',
            'Genome',
            'Protein Classification',
            
            # positional features
            'gene_number',
            'contig_pos_start',
            'contig_pos_end',
            'length',
            'frame',
            'contig_left_end_dist', 'contig_right_end_dist',
            'contig_left_end_gene_dist', 'contig_right_end_gene_dist',
            
            # raw scores
            'Pfam_V-score', 'Pfam_VL-score', 'KEGG_V-score', 'KEGG_VL-score',
            'PHROG_V-score', 'PHROG_VL-score',

            # contig / window aggregates
            'contig_avg_KEGG_V-score', 'contig_avg_KEGG_VL-score',
            'contig_avg_Pfam_V-score', 'contig_avg_Pfam_VL-score',
            'contig_avg_PHROG_V-score', 'contig_avg_PHROG_VL-score',
            'window_avg_KEGG_VL-score', 'window_avg_Pfam_VL-score', 'window_avg_PHROG_VL-score',

            # viral / MGE distances
            'KEGG_viral_left_dist', 'KEGG_viral_right_dist',
            'Pfam_viral_left_dist', 'Pfam_viral_right_dist',
            'PHROG_viral_left_dist', 'PHROG_viral_right_dist',
            'KEGG_MGE_left_dist', 'KEGG_MGE_right_dist',
            'Pfam_MGE_left_dist', 'Pfam_MGE_right_dist',
            'PHROG_MGE_left_dist', 'PHROG_MGE_right_dist',

            # V/VL-scores of nearest left/right MGE gene
            'KEGG_V-score_left_MGE', 'KEGG_V-score_right_MGE',
            'KEGG_VL-score_left_MGE', 'KEGG_VL-score_right_MGE',
            'Pfam_V-score_left_MGE', 'Pfam_V-score_right_MGE',
            'Pfam_VL-score_left_MGE', 'Pfam_VL-score_right_MGE',
            'PHROG_V-score_left_MGE', 'PHROG_V-score_right_MGE',
            'PHROG_VL-score_left_MGE', 'PHROG_VL-score_right_MGE',
        
            # other
            'circular_contig',
    
            # LGBM results (circular but just for inspection, not training)
            'LGBM_viral_prob',
            'Viral_Origin_Confidence',

            # Context-based features
            'viral_region_id',
            'step5_in_merged_region',

            # labels
            'region_label',
            'region_contig_type',
            'True Positive',
            'True Negative',
            'Source',            
        ]
    ).sort(
        ["Contig", "contig_pos_start", "contig_pos_end"]
    )

    if modify_labels:
        # 5) Modify region labels based on context-based features and window average VL scores
        final_lazy = (
            final_lazy
            .with_columns(
                pl.col("region_label").alias("original_region_label"),
            )
            .with_columns(
                pl.when(
                    # Protein is NOT in a refined viral region
                    ~(pl.col("step5_in_merged_region").fill_null(False))
                )
                .then(pl.lit("Nonviral"))
                .otherwise(pl.col("original_region_label"))
                .alias("region_label")
            )
            .with_columns(
                (pl.col("region_label") == "Viral").alias("True Positive"),
                (pl.col("region_label") != "Viral").alias("True Negative"),
            )
            .unique()
            .sort(["Contig", "contig_pos_start", "contig_pos_end"])
        )
    else:
        final_lazy = (
            final_lazy
            .with_columns(
                pl.col("region_label").alias("original_region_label"),
            )
            .unique()
            .sort(["Contig", "contig_pos_start", "contig_pos_end"])
        )
    
    # 6) Execute everything
    return final_lazy.collect()

In [39]:
CHECKAMG_FINAL_RESULTS_GENOMAD = CHECKAMG_OUTPUT_DIR.joinpath("checkamg_annotate_genomad_dataset/results/final_results.parquet")
CHECKAMG_GENOMIC_CONTEXT_GENOMAD = CHECKAMG_OUTPUT_DIR.joinpath("checkamg_annotate_genomad_dataset/results/genes_genomic_context.parquet")

In [40]:
genomad_training_df = (
    load_data(
        CHECKAMG_FINAL_RESULTS_GENOMAD,
        CHECKAMG_GENOMIC_CONTEXT_GENOMAD,
        GENOMAD_REGIONS_OUTPUT,
        modify_labels=False
        )
    .with_columns(pl.lit("geNomad").alias("Dataset"))
)

In [41]:
genomad_training_df

Protein,Contig,Genome,Protein Classification,gene_number,contig_pos_start,contig_pos_end,length,frame,contig_left_end_dist,contig_right_end_dist,contig_left_end_gene_dist,contig_right_end_gene_dist,Pfam_V-score,Pfam_VL-score,KEGG_V-score,KEGG_VL-score,PHROG_V-score,PHROG_VL-score,contig_avg_KEGG_V-score,contig_avg_KEGG_VL-score,contig_avg_Pfam_V-score,contig_avg_Pfam_VL-score,contig_avg_PHROG_V-score,contig_avg_PHROG_VL-score,window_avg_KEGG_VL-score,window_avg_Pfam_VL-score,window_avg_PHROG_VL-score,KEGG_viral_left_dist,KEGG_viral_right_dist,Pfam_viral_left_dist,Pfam_viral_right_dist,PHROG_viral_left_dist,PHROG_viral_right_dist,KEGG_MGE_left_dist,KEGG_MGE_right_dist,Pfam_MGE_left_dist,Pfam_MGE_right_dist,PHROG_MGE_left_dist,PHROG_MGE_right_dist,KEGG_V-score_left_MGE,KEGG_V-score_right_MGE,KEGG_VL-score_left_MGE,KEGG_VL-score_right_MGE,Pfam_V-score_left_MGE,Pfam_V-score_right_MGE,Pfam_VL-score_left_MGE,Pfam_VL-score_right_MGE,PHROG_V-score_left_MGE,PHROG_V-score_right_MGE,PHROG_VL-score_left_MGE,PHROG_VL-score_right_MGE,circular_contig,LGBM_viral_prob,Viral_Origin_Confidence,viral_region_id,step5_in_merged_region,region_label,region_contig_type,True Positive,True Negative,Source,original_region_label,Dataset
str,str,str,str,i64,i64,i64,i64,i64,f32,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,bool,f32,str,i32,bool,str,str,bool,bool,str,str,str
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_1""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""unclassified""",1,36,197,161,1,35.0,4906.0,0.0,7.0,null,null,null,null,null,null,8.356667,3.563338,8.7175,3.90865,8.173333,3.166663,3.563338,3.90865,3.166663,null,381.0,null,162.0,null,381.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,0.884765,"""medium""",null,false,"""Viral""","""standalone_virus""",true,false,"""Virus""","""Viral""","""geNomad"""
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_2""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""unclassified""",2,194,412,218,-1,193.0,4691.0,1.0,6.0,10.0,4.823363,null,null,null,null,8.356667,3.563338,8.7175,3.90865,8.173333,3.166663,3.563338,3.90865,3.166663,null,219.0,null,219.0,null,219.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,0.708898,"""medium""",null,false,"""Viral""","""standalone_virus""",true,false,"""Virus""","""Viral""","""geNomad"""
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_3""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""unclassified""",3,520,2157,1637,1,519.0,2946.0,2.0,5.0,10.0,4.473399,10.0,4.484769,10.0,3.401745,8.356667,3.563338,8.7175,3.90865,8.173333,3.166663,3.563338,3.90865,3.166663,null,1638.0,219.0,1638.0,null,1638.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,0.77663,"""medium""",null,false,"""Viral""","""standalone_virus""",true,false,"""Virus""","""Viral""","""geNomad"""
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_4""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""unclassified""",4,2154,2429,275,1,2153.0,2674.0,3.0,4.0,10.0,3.650308,10.0,3.500236,10.0,3.443106,8.356667,3.563338,8.7175,3.90865,8.173333,3.166663,3.563338,3.90865,3.166663,1638.0,null,1638.0,null,1638.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,0.950381,"""high""",null,false,"""Viral""","""standalone_virus""",true,false,"""Virus""","""Viral""","""geNomad"""
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_5""","""1015567_contig_2210_74_1015567_contig_2210|frag

In [42]:
genomad_training_df.get_column("Protein").is_duplicated().sum()

0

In [43]:
CHECKAMG_FINAL_RESULTS_PROGENOMES = "/storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/checkamg_annotate_outputs_1.1/checkamg_annotate_progenomes_dataset/results/final_results.parquet"
CHECKAMG_GENOMIC_CONTEXT_PROGENOMES = "/storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/checkamg_annotate_outputs_1.1/checkamg_annotate_progenomes_dataset/results/genes_genomic_context.parquet"

In [44]:
progenomes_training_df = (
    load_data(
        CHECKAMG_FINAL_RESULTS_PROGENOMES,
        CHECKAMG_GENOMIC_CONTEXT_PROGENOMES,
        PROGENOMES_REGIONS_OUTPUT,
        modify_labels=True
        )
    .with_columns(pl.lit("progenomes").alias("Dataset"))
    )

In [45]:
progenomes_training_df

Protein,Contig,Genome,Protein Classification,gene_number,contig_pos_start,contig_pos_end,length,frame,contig_left_end_dist,contig_right_end_dist,contig_left_end_gene_dist,contig_right_end_gene_dist,Pfam_V-score,Pfam_VL-score,KEGG_V-score,KEGG_VL-score,PHROG_V-score,PHROG_VL-score,contig_avg_KEGG_V-score,contig_avg_KEGG_VL-score,contig_avg_Pfam_V-score,contig_avg_Pfam_VL-score,contig_avg_PHROG_V-score,contig_avg_PHROG_VL-score,window_avg_KEGG_VL-score,window_avg_Pfam_VL-score,window_avg_PHROG_VL-score,KEGG_viral_left_dist,KEGG_viral_right_dist,Pfam_viral_left_dist,Pfam_viral_right_dist,PHROG_viral_left_dist,PHROG_viral_right_dist,KEGG_MGE_left_dist,KEGG_MGE_right_dist,Pfam_MGE_left_dist,Pfam_MGE_right_dist,PHROG_MGE_left_dist,PHROG_MGE_right_dist,KEGG_V-score_left_MGE,KEGG_V-score_right_MGE,KEGG_VL-score_left_MGE,KEGG_VL-score_right_MGE,Pfam_V-score_left_MGE,Pfam_V-score_right_MGE,Pfam_VL-score_left_MGE,Pfam_VL-score_right_MGE,PHROG_V-score_left_MGE,PHROG_V-score_right_MGE,PHROG_VL-score_left_MGE,PHROG_VL-score_right_MGE,circular_contig,LGBM_viral_prob,Viral_Origin_Confidence,viral_region_id,step5_in_merged_region,region_label,region_contig_type,True Positive,True Negative,Source,original_region_label,Dataset
str,str,str,str,i64,i64,i64,i64,i64,f32,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,bool,f32,str,i32,bool,str,str,bool,bool,str,str,str
"""1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge_1""","""1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge""","""1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge""","""unclassified""",1,3,521,518,-1,2.0,76095.0,0.0,62.0,10.0,3.732072,10.0,3.571243,10.0,3.851747,3.554286,2.113305,3.468163,2.027925,4.3376,2.476098,2.448551,2.203206,3.083517,null,519.0,null,519.0,null,519.0,null,45087.0,null,null,null,45087.0,null,10.0,null,3.75473,null,null,null,null,null,10.0,null,3.078457,false,0.003014,"""low""",0,true,"""Nonviral""","""standalone_mge""",false,true,"""MGE""","""Nonviral""","""progenomes"""
"""1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge_2""","""1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge""","""1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge""","""metabolic""",2,518,1300,782,-1,517.0,75316.0,1.0,61.0,10.0,3.326745,10.0,3.502564,10.0,3.535041,3.554286,2.113305,3.468163,2.027925,4.3376,2.476098,2.448551,2.203206,3.083517,519.0,4437.0,519.0,4437.0,519.0,4437.0,null,44568.0,null,null,null,44568.0,null,10.0,null,3.75473,null,null,null,null,null,10.0,null,3.078457,false,0.003774,"""low""",0,true,"""Nonviral""","""standalone_mge""",false,true,"""MGE""","""Nonviral""","""progenomes"""
"""1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge_3""","""1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge""","""1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge""","""metabolic""",3,1407,1946,539,-1,1406.0,74670.0,2.0,60.0,0.12,1.079181,0.12,1.079181,null,null,3.554286,2.113305,3.468163,2.027925,4.3376,2.476098,2.448551,2.203206,3.083517,783.0,3654.0,783.0,3654.0,783.0,3654.0,null,43785.0,null,null,null,43785.0,null,10.0,null,3.75473,null,null,null,null,null,10.0,null,3.078457,false,0.001655,"""low""",0,true,"""Nonviral""","""standalone_mge""",false,true,"""MGE""","""Nonviral""","""progenomes"""
"""1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge_4""","""1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge""","""1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge""","""unclassified""",4,1951,2262,311,-1,1950.0,74354.0,3.0,59.0,0.35,1.544068,null,null,null,null,3.554286,2.113305,3.468163,2.027925,4.3376,2.476098,2.448551,2.203206,3.083517,1323.0,3114.0,1323.0,3114.0,1323.0,3114.0,null,43245.0,null,null,null,43245.0,null,10.0,null,3.75473,null,null,null,null,null,10.0,null,3.078457,false,0.002001,"""low""",0,true,"""Nonviral""","""standalone_mge""",false,true,"""MGE""","""Nonviral"""

In [46]:
progenomes_training_df.get_column("Protein").is_duplicated().sum()

0

Inspect cases where the region label was modified

In [47]:
progenomes_training_df.filter(pl.col("region_label") != pl.col("original_region_label"))

Protein,Contig,Genome,Protein Classification,gene_number,contig_pos_start,contig_pos_end,length,frame,contig_left_end_dist,contig_right_end_dist,contig_left_end_gene_dist,contig_right_end_gene_dist,Pfam_V-score,Pfam_VL-score,KEGG_V-score,KEGG_VL-score,PHROG_V-score,PHROG_VL-score,contig_avg_KEGG_V-score,contig_avg_KEGG_VL-score,contig_avg_Pfam_V-score,contig_avg_Pfam_VL-score,contig_avg_PHROG_V-score,contig_avg_PHROG_VL-score,window_avg_KEGG_VL-score,window_avg_Pfam_VL-score,window_avg_PHROG_VL-score,KEGG_viral_left_dist,KEGG_viral_right_dist,Pfam_viral_left_dist,Pfam_viral_right_dist,PHROG_viral_left_dist,PHROG_viral_right_dist,KEGG_MGE_left_dist,KEGG_MGE_right_dist,Pfam_MGE_left_dist,Pfam_MGE_right_dist,PHROG_MGE_left_dist,PHROG_MGE_right_dist,KEGG_V-score_left_MGE,KEGG_V-score_right_MGE,KEGG_VL-score_left_MGE,KEGG_VL-score_right_MGE,Pfam_V-score_left_MGE,Pfam_V-score_right_MGE,Pfam_VL-score_left_MGE,Pfam_VL-score_right_MGE,PHROG_V-score_left_MGE,PHROG_V-score_right_MGE,PHROG_VL-score_left_MGE,PHROG_VL-score_right_MGE,circular_contig,LGBM_viral_prob,Viral_Origin_Confidence,viral_region_id,step5_in_merged_region,region_label,region_contig_type,True Positive,True Negative,Source,original_region_label,Dataset
str,str,str,str,i64,i64,i64,i64,i64,f32,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,bool,f32,str,i32,bool,str,str,bool,bool,str,str,str
"""1000562.SAMN03114893.JSAP01000059~167-104902~standalone_virus_1""","""1000562.SAMN03114893.JSAP01000059~167-104902~standalone_virus""","""1000562.SAMN03114893.JSAP01000059~167-104902~standalone_virus""","""unclassified""",1,3,542,539,-1,2.0,104193.0,0.0,134.0,0.02,0.30103,0.07,0.845098,0.03,0.477121,3.715079,2.248084,5.165814,2.544419,5.918,2.629816,1.356998,2.107686,1.229696,null,20769.0,null,3888.0,null,21477.0,null,51348.0,null,null,null,51348.0,null,10.0,null,4.882297,null,null,null,null,null,10.0,null,4.578307,false,0.000023,"""low""",null,false,"""Nonviral""","""standalone_virus""",false,true,"""Virus""","""Viral""","""progenomes"""
"""1000562.SAMN03114893.JSAP01000059~167-104902~standalone_virus_2""","""1000562.SAMN03114893.JSAP01000059~167-104902~standalone_virus""","""1000562.SAMN03114893.JSAP01000059~167-104902~standalone_virus""","""unclassified""",2,1034,2230,1196,-1,1033.0,102505.0,1.0,133.0,2.32,2.365488,0.15,1.176091,null,null,3.715079,2.248084,5.165814,2.544419,5.918,2.629816,1.399629,2.023415,1.229696,null,20229.0,null,3348.0,null,20937.0,null,50808.0,null,null,null,50808.0,null,10.0,null,4.882297,null,null,null,null,null,10.0,null,4.578307,false,0.000118,"""low""",null,false,"""Nonviral""","""standalone_virus""",false,true,"""Virus""","""Viral""","""progenomes"""
"""1000562.SAMN03114893.JSAP01000059~167-104902~standalone_virus_3""","""1000562.SAMN03114893.JSAP01000059~167-104902~standalone_virus""","""1000562.SAMN03114893.JSAP01000059~167-104902~standalone_virus""","""unclassified""",3,2254,2856,602,-1,2253.0,101879.0,2.0,132.0,2.32,2.365488,0.15,1.176091,null,null,3.715079,2.248084,5.165814,2.544419,5.918,2.629816,1.399629,2.023415,1.229696,null,19032.0,null,2151.0,null,19740.0,null,49611.0,null,null,null,49611.0,null,10.0,null,4.882297,null,null,null,null,null,10.0,null,4.578307,false,0.000122,"""low""",null,false,"""Nonviral""","""standalone_virus""",false,true,"""Virus""","""Viral""","""progenomes"""
"""1000562.SAMN03114893.JSAP01000059~167-104902~standalone_virus_4""","""1000562.SAMN03114893.JSAP01000059~167-104902~standalone_virus""","""1000562.SAMN03114893.JSAP01000059~167-104902~standalone_virus""","""unclassified""",4,2874,3827,953,-1,2873.0,100908.0,3.0,131.0,0.09,0.954243,0.09,0.954243,null,null,3.715079,2.248084,5.165814,2.544419,5.918,2.629816,1.532308,2.066689,1.388988,null,18429.0,null,1548.0,null,19137.0,null,49008.0,null,null,null,49008.0,null,10.0,null,4.882297,null,null,null,null,null,10.0,null,4.578307,false,0.000099,"""low""",

### Combine the two two pre-training dataframes into one

In [48]:
training_df_pre = pl.concat([genomad_training_df, progenomes_training_df], how="vertical")

In [49]:
training_df_pre

Protein,Contig,Genome,Protein Classification,gene_number,contig_pos_start,contig_pos_end,length,frame,contig_left_end_dist,contig_right_end_dist,contig_left_end_gene_dist,contig_right_end_gene_dist,Pfam_V-score,Pfam_VL-score,KEGG_V-score,KEGG_VL-score,PHROG_V-score,PHROG_VL-score,contig_avg_KEGG_V-score,contig_avg_KEGG_VL-score,contig_avg_Pfam_V-score,contig_avg_Pfam_VL-score,contig_avg_PHROG_V-score,contig_avg_PHROG_VL-score,window_avg_KEGG_VL-score,window_avg_Pfam_VL-score,window_avg_PHROG_VL-score,KEGG_viral_left_dist,KEGG_viral_right_dist,Pfam_viral_left_dist,Pfam_viral_right_dist,PHROG_viral_left_dist,PHROG_viral_right_dist,KEGG_MGE_left_dist,KEGG_MGE_right_dist,Pfam_MGE_left_dist,Pfam_MGE_right_dist,PHROG_MGE_left_dist,PHROG_MGE_right_dist,KEGG_V-score_left_MGE,KEGG_V-score_right_MGE,KEGG_VL-score_left_MGE,KEGG_VL-score_right_MGE,Pfam_V-score_left_MGE,Pfam_V-score_right_MGE,Pfam_VL-score_left_MGE,Pfam_VL-score_right_MGE,PHROG_V-score_left_MGE,PHROG_V-score_right_MGE,PHROG_VL-score_left_MGE,PHROG_VL-score_right_MGE,circular_contig,LGBM_viral_prob,Viral_Origin_Confidence,viral_region_id,step5_in_merged_region,region_label,region_contig_type,True Positive,True Negative,Source,original_region_label,Dataset
str,str,str,str,i64,i64,i64,i64,i64,f32,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,bool,f32,str,i32,bool,str,str,bool,bool,str,str,str
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_1""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""unclassified""",1,36,197,161,1,35.0,4906.0,0.0,7.0,null,null,null,null,null,null,8.356667,3.563338,8.7175,3.90865,8.173333,3.166663,3.563338,3.90865,3.166663,null,381.0,null,162.0,null,381.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,0.884765,"""medium""",null,false,"""Viral""","""standalone_virus""",true,false,"""Virus""","""Viral""","""geNomad"""
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_2""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""unclassified""",2,194,412,218,-1,193.0,4691.0,1.0,6.0,10.0,4.823363,null,null,null,null,8.356667,3.563338,8.7175,3.90865,8.173333,3.166663,3.563338,3.90865,3.166663,null,219.0,null,219.0,null,219.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,0.708898,"""medium""",null,false,"""Viral""","""standalone_virus""",true,false,"""Virus""","""Viral""","""geNomad"""
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_3""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""unclassified""",3,520,2157,1637,1,519.0,2946.0,2.0,5.0,10.0,4.473399,10.0,4.484769,10.0,3.401745,8.356667,3.563338,8.7175,3.90865,8.173333,3.166663,3.563338,3.90865,3.166663,null,1638.0,219.0,1638.0,null,1638.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,0.77663,"""medium""",null,false,"""Viral""","""standalone_virus""",true,false,"""Virus""","""Viral""","""geNomad"""
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_4""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""unclassified""",4,2154,2429,275,1,2153.0,2674.0,3.0,4.0,10.0,3.650308,10.0,3.500236,10.0,3.443106,8.356667,3.563338,8.7175,3.90865,8.173333,3.166663,3.563338,3.90865,3.166663,1638.0,null,1638.0,null,1638.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,0.950381,"""high""",null,false,"""Viral""","""standalone_virus""",true,false,"""Virus""","""Viral""","""geNomad"""
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_5""","""1015567_contig_2210_74_1015567_contig_2210|frag

In [50]:
training_df_pre.filter(pl.col("Protein").is_duplicated())

Protein,Contig,Genome,Protein Classification,gene_number,contig_pos_start,contig_pos_end,length,frame,contig_left_end_dist,contig_right_end_dist,contig_left_end_gene_dist,contig_right_end_gene_dist,Pfam_V-score,Pfam_VL-score,KEGG_V-score,KEGG_VL-score,PHROG_V-score,PHROG_VL-score,contig_avg_KEGG_V-score,contig_avg_KEGG_VL-score,contig_avg_Pfam_V-score,contig_avg_Pfam_VL-score,contig_avg_PHROG_V-score,contig_avg_PHROG_VL-score,window_avg_KEGG_VL-score,window_avg_Pfam_VL-score,window_avg_PHROG_VL-score,KEGG_viral_left_dist,KEGG_viral_right_dist,Pfam_viral_left_dist,Pfam_viral_right_dist,PHROG_viral_left_dist,PHROG_viral_right_dist,KEGG_MGE_left_dist,KEGG_MGE_right_dist,Pfam_MGE_left_dist,Pfam_MGE_right_dist,PHROG_MGE_left_dist,PHROG_MGE_right_dist,KEGG_V-score_left_MGE,KEGG_V-score_right_MGE,KEGG_VL-score_left_MGE,KEGG_VL-score_right_MGE,Pfam_V-score_left_MGE,Pfam_V-score_right_MGE,Pfam_VL-score_left_MGE,Pfam_VL-score_right_MGE,PHROG_V-score_left_MGE,PHROG_V-score_right_MGE,PHROG_VL-score_left_MGE,PHROG_VL-score_right_MGE,circular_contig,LGBM_viral_prob,Viral_Origin_Confidence,viral_region_id,step5_in_merged_region,region_label,region_contig_type,True Positive,True Negative,Source,original_region_label,Dataset
str,str,str,str,i64,i64,i64,i64,i64,f32,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,bool,f32,str,i32,bool,str,str,bool,bool,str,str,str


In [51]:
training_df_pre = training_df_pre.unique().sort(["Dataset", "Contig", "contig_pos_start", "contig_pos_end"])

In [52]:
GENOMAD_PROGENOMES_COMBINED = ROOT_DIR.joinpath("processing/genomad_progenomes_combined")

In [53]:
! mkdir -p {GENOMAD_PROGENOMES_COMBINED}

In [54]:
training_df_pre.write_parquet(
    GENOMAD_PROGENOMES_COMBINED.joinpath("training_data_pre_split.parquet")
)